# 04 · Validación de la transformación de features

**Pipeline corregido**

1. Features crudas (≥ 0), una fila por (sujeto, estímulo)
2. **Transformación global** (Yeo-Johnson o Box-Cox): un λ por columna (feature × canal), ajustado con todos los sujetos → corrige la **forma** (asimetría)
3. **Z-score por (subject, origin_db)** → elimina **nivel y escala** propios de cada sujeto

**Qué cambió respecto a la versión anterior**

| Antes | Ahora | Motivo |
|---|---|---|
| `PowerTransformer` ajustado **por sujeto** | Ajustado **global** | Con `df_mean` cada sujeto tiene ~8 filas: se estimaban 64 λ con 8 puntos cada uno. Además cada sujeto quedaba con una curva distinta, por lo que la distancia entre agarres no era comparable entre sujetos. |
| `standardize=True` dentro del transformador | `standardize=False` + z-score por sujeto explícito | Separar forma (global) de nivel/escala (por sujeto). |
| Box-Cox hacía `continue` si había valores ≤ 0 | Error explícito con las columnas afectadas | El `continue` dejaba a ese sujeto **sin transformar** y mezclado con los demás. |
| Validación con `std(ddof=1)` sobre `standardize=True` (ddof=0) | Todo con `ddof=1` | La std de 1.069 = √(8/7) era ese desajuste, no un error de datos. |
| Reposo incluido en el ajuste | Parámetro `INCLUIR_REPOSO` | Las filas de reposo son 100–300× mayores y dominan λ, media y std (ver diagnóstico). |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.lines import Line2D
from sklearn.preprocessing import PowerTransformer

# Configuración

In [ ]:
#RUTA = '../data/processed/df_max.xlsx'
RUTA = '../data/processed/df_mean.xlsx'
OUT_DIR = Path('../data/processed')

GROUP_COLS = ['subject', 'origin_db']   # unidad de normalización
ESTIMULO_REPOSO = 0

# False: se excluye el reposo del ajuste y de los datos exportados.
# Motivo: sus valores son ~100x mayores que los agarres (ver diagnóstico) y,
# con ~8 filas por sujeto, dominan λ, la media y la std del z-score.
INCLUIR_REPOSO = False

features = ['VAR_STD', 'Range', 'Energy', 'Peak_to_Peak', 'VAR', 'Max', 'MNF', 'SSC']
signal_cols = [f'Channel {ch}_{feat}' for ch in range(1, 9) for feat in features]

# Cargar datos

In [ ]:
intact_all = pd.read_excel(RUTA)

if INCLUIR_REPOSO:
    intact = intact_all.copy()
else:
    intact = intact_all[intact_all['stimulus'] != ESTIMULO_REPOSO].copy()

intact = intact.reset_index(drop=True)
print(f'Filas totales: {len(intact_all)}  ->  usadas: {len(intact)}  (INCLUIR_REPOSO={INCLUIR_REPOSO})')
intact.head(10)

# Diagnóstico previo

Antes de transformar hay que saber:
- Cuántas filas tiene cada sujeto (define si algo "por sujeto" es estimable).
- Si hay ceros o negativos (Box-Cox exige > 0).
- En qué escala está cada feature (Yeo-Johnson con datos ≥ 0 equivale a Box-Cox sobre x + 1, y pierde fuerza si los valores son diminutos).
- Cuánto se aleja el reposo de los agarres.

In [ ]:
filas_por_grupo = intact.groupby(GROUP_COLS).size()
print('Filas por (subject, origin_db):')
print(filas_por_grupo.describe().round(2))

In [ ]:
def resumen_escala(df, cols):
    X = df[cols]
    return pd.DataFrame({
        'min': X.min(),
        'mediana': X.median(),
        'max': X.max(),
        'pct_ceros': (X == 0).mean() * 100,
        'pct_negativos': (X < 0).mean() * 100,
    })

escala = resumen_escala(intact, signal_cols)

# Resumen por tipo de feature (agregando los 8 canales)
escala_feat = escala.copy()
escala_feat['feature'] = [c.split('_', 1)[1] for c in escala_feat.index]
print(escala_feat.groupby('feature').agg(
    min=('min', 'min'), mediana=('mediana', 'median'), max=('max', 'max'),
    pct_ceros=('pct_ceros', 'max'), pct_negativos=('pct_negativos', 'max')
).loc[features].round(6))

cols_no_positivas = escala.index[escala['min'] <= 0].tolist()
print('\nColumnas con valores <= 0 (Box-Cox no aplicable):', cols_no_positivas or 'ninguna')

In [ ]:
# ¿Cuánto más grande es el reposo que los agarres? (mediana reposo / mediana agarres)
reposo = intact_all[intact_all['stimulus'] == ESTIMULO_REPOSO]
agarres = intact_all[intact_all['stimulus'] != ESTIMULO_REPOSO]

ratio = (reposo[signal_cols].median() / agarres[signal_cols].median())
ratio_feat = ratio.groupby([c.split('_', 1)[1] for c in ratio.index]).median().loc[features]
print('Mediana(reposo) / Mediana(agarres), por feature:')
print(ratio_feat.round(1))

> **Revisar la extracción del reposo.** En la versión anterior, las filas de reposo mostraban MNF ≈ 17 000–34 000 Hz y SSC ≈ 10 000,
> frente a ~100 en los agarres. Ninguna base de NinaPro tiene una frecuencia de Nyquist tan alta (máximo 1 kHz con 2 kHz de muestreo),
> así que es muy probable que el reposo se haya calculado sobre el segmento completo en lugar de por ventanas de 200 ms,
> o con otra agregación. Mientras eso no se corrija, conviene dejar `INCLUIR_REPOSO = False`.
>
> También vale la pena verificar la `fs` usada para MNF en DB1 (Otto Bock, 100 Hz): MNF ≈ 100–130 Hz superaría su Nyquist (50 Hz).

# Funciones del pipeline

In [ ]:
def zscore_por_sujeto(df, cols, group_cols=GROUP_COLS):
    '''
    Z-score por (subject, origin_db), con ddof=1.

    Casos especiales:
    - Menos de 2 muestras -> se asigna 0.
    - STD = 0 o NaN -> se usa 1 para evitar divisiones por cero.
    '''
    df_res = df.copy()
    df_res[cols] = df_res[cols].astype(float)

    for _, group in df.groupby(group_cols):
        idx = group.index
        sub = df_res.loc[idx, cols].values

        if sub.shape[0] < 2:
            df_res.loc[idx, cols] = 0.0
            continue

        means = np.nanmean(sub, axis=0)
        stds = np.nanstd(sub, axis=0, ddof=1)
        stds[(stds == 0) | np.isnan(stds)] = 1.0

        z = (sub - means) / stds
        df_res.loc[idx, cols] = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)

    return df_res


def ajustar_transformacion_global(df_fit, cols, method):
    '''
    Ajusta UN PowerTransformer global (un λ por columna).
    standardize=False: la estandarización se hace después, por sujeto.

    En LOSO (notebook 09) df_fit debe contener SOLO los sujetos de entrenamiento.
    '''
    X = df_fit[cols].astype(float)

    if method == 'box-cox':
        malas = X.columns[(X <= 0).any()].tolist()
        if malas:
            raise ValueError(f'Box-Cox requiere valores > 0. Columnas con valores <= 0: {malas}')

    pt = PowerTransformer(method=method, standardize=False)
    pt.fit(X.values)
    return pt


def aplicar_transformacion(df, pt, cols):
    df_res = df.copy()
    df_res[cols] = pt.transform(df[cols].astype(float).values)
    return df_res


def pipeline_transformacion(df, cols, method, df_fit=None):
    '''
    1) Transformación global  2) Z-score por sujeto.
    Devuelve (df_transformado_sin_z, df_final, transformador).
    '''
    pt = ajustar_transformacion_global(df if df_fit is None else df_fit, cols, method)
    df_t = aplicar_transformacion(df, pt, cols)
    df_final = zscore_por_sujeto(df_t, cols)
    return df_t, df_final, pt


def tabla_lambdas(pt, cols):
    lam = pd.DataFrame({'col': cols, 'lambda': pt.lambdas_})
    lam['channel'] = lam['col'].str.extract(r'Channel (\d+)_')[0].astype(int)
    lam['feature'] = lam['col'].str.split('_', n=1).str[1]
    return lam.pivot(index='feature', columns='channel', values='lambda').loc[features]

# Funciones de validación

In [ ]:
def validate_normalization(df_normalized, cols, group_cols=GROUP_COLS):
    '''Media y std (ddof=1) por grupo tras la normalización.'''
    rows = []
    for (subj, db), group in df_normalized.groupby(group_cols):
        means = group[cols].mean()
        stds = group[cols].std(ddof=1)
        for col in cols:
            rows.append({'subject': subj, 'db': db, 'signal': col,
                         'mean_after': means[col], 'std_after': stds[col]})
    return pd.DataFrame(rows)


def resumen_validacion(df_normalized, cols, nombre):
    v = validate_normalization(df_normalized, cols)
    # std = 0 legítima: columnas constantes dentro de un sujeto (quedan en 0)
    fuera_media = (v['mean_after'].abs() > 0.01).sum()
    fuera_std = ((v['std_after'] - 1).abs() > 0.01).sum()
    print(f'[{nombre}] NaN: {df_normalized[cols].isna().sum().sum()} | '
          f'grupos con |media| > 0.01: {fuera_media} | grupos con |std-1| > 0.01: {fuera_std}')
    return v


def check_transform_integrity(df_raw, df_t, df_final, cols, group_cols=GROUP_COLS):
    '''
    Chequeos de invarianza de rangos (Spearman):
    A) Transformación global (monótona): la matriz de Spearman agregada debe ser idéntica a la cruda.
    B) Transformación + z-score por sujeto: dentro de cada sujeto el orden de cada columna no cambia.
    Cualquier diferencia indica un problema de datos (p. ej. CSV desactualizados o filas desalineadas).
    '''
    sp_raw = df_raw[cols].corr(method='spearman')
    sp_t = df_t[cols].corr(method='spearman')
    max_diff_A = (sp_raw - sp_t).abs().max().max()

    fallos_B = []
    for key, g in df_raw.groupby(group_cols):
        r_raw = g[cols].rank()
        r_fin = df_final.loc[g.index, cols].rank()
        malas = r_raw.columns[(r_raw != r_fin).any()]
        # columnas constantes dentro del grupo quedan en 0: siguen empatadas, no cuentan como fallo
        fallos_B += [(key, c) for c in malas]

    print(f'A) max |ΔSpearman| global (crudo vs transformado): {max_diff_A:.2e}')
    print(f'B) columnas con orden alterado dentro de algún sujeto: {len(fallos_B)}')
    ok = (max_diff_A < 1e-10) and not fallos_B
    print('OK' if ok else 'REVISAR')
    return ok

# Funciones de visualización

In [ ]:
def plot_box_before_after(df_original, df_norm, cols, titulo_norm):
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    df_original[cols].boxplot(ax=axes[0], rot=90)
    axes[0].set_title('Datos originales')
    df_norm[cols].boxplot(ax=axes[1], rot=90)
    axes[1].set_title(titulo_norm)
    axes[1].set_ylim(-5, 5)
    for ax in axes:
        ax.tick_params(axis='x', labelsize=6)
    plt.tight_layout(); plt.show()


def plot_histograms_by_channel(df, title):
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(title, fontsize=16)
    for i, feat in enumerate(features):
        ax = axes[i // 4][i % 4]
        vals = df[[f'Channel {ch}_{feat}' for ch in range(1, 9)]].values.flatten()
        sns.histplot(vals, kde=True, ax=ax, bins=20, color='steelblue')
        ax.set_title(feat, fontsize=12)
        ax.set_xlabel('')
    plt.tight_layout(); plt.show()


def plot_kde_normalized(df_norm, features):
    stimuli = sorted(df_norm['stimulus'].unique())
    colors = sns.color_palette('Set2' if len(stimuli) <= 8 else 'husl', len(stimuli))
    palette = dict(zip(stimuli, colors))

    for feature in features:
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        for ch in range(1, 9):
            ax = axes[(ch - 1) // 4][(ch - 1) % 4]
            col = f'Channel {ch}_{feature}'
            for stim in stimuli:
                vals = df_norm.loc[df_norm['stimulus'] == stim, col]
                sns.kdeplot(x=vals, ax=ax, color=palette[stim], linewidth=2, legend=False, warn_singular=False)
            ax.axvline(0, color='black', linestyle='--', alpha=0.5)
            ax.set_xlim(-4, 4)
            ax.set_title(f'Channel {ch}', fontsize=11)
            ax.set_xlabel(''); ax.set_ylabel('')

        handles = [Line2D([0], [0], color=palette[s], lw=2) for s in stimuli]
        fig.suptitle(f'{feature} (transformación global + Z-score por sujeto)', fontsize=16, y=0.99)
        fig.legend(handles, stimuli, title='Estímulo', loc='upper center',
                   bbox_to_anchor=(0.5, 0.95), ncol=min(len(stimuli), 10), frameon=False)
        plt.tight_layout(rect=[0, 0, 1, 0.88]); plt.show()

# Línea base: solo Z-score por sujeto

Se conserva como referencia para ver cuánto aporta la transformación.

In [ ]:
intact_norm = zscore_por_sujeto(intact, signal_cols)
v_norm = resumen_validacion(intact_norm, signal_cols, 'Solo z-score')
intact_norm.to_csv(OUT_DIR / 'intact_norm.csv', index=False)

# Yeo-Johnson global → Z-score por sujeto

Para EDA el transformador se ajusta con todos los sujetos. Para el benchmark LOSO se reutiliza `pipeline_transformacion(df_test, cols, method, df_fit=df_train)`
dentro de cada fold, para no filtrar información del sujeto de prueba.

In [ ]:
intact_yj_t, intact_yj, pt_yj = pipeline_transformacion(intact, signal_cols, 'yeo-johnson')

v_yj = resumen_validacion(intact_yj, signal_cols, 'Yeo-Johnson -> z')
check_transform_integrity(intact, intact_yj_t, intact_yj, signal_cols)

intact_yj.to_csv(OUT_DIR / 'intact_yj.csv', index=False)

### Todos los estímulos por canal

In [ ]:
plot_kde_normalized(intact_yj, features)

### Histogramas

In [ ]:
plot_histograms_by_channel(intact_yj, 'Yeo-Johnson global + Z-score por sujeto')

# Box-Cox global → Z-score por sujeto

Solo es aplicable si todas las columnas son estrictamente positivas (ver diagnóstico). Si alguna tiene ceros, la función lanza un error
indicando cuáles, en lugar de dejar sujetos sin transformar.

In [ ]:
if cols_no_positivas:
    print('Box-Cox omitido: hay columnas con valores <= 0 ->', cols_no_positivas)
    intact_bc_t, intact_bc, pt_bc = None, None, None
else:
    intact_bc_t, intact_bc, pt_bc = pipeline_transformacion(intact, signal_cols, 'box-cox')
    v_bc = resumen_validacion(intact_bc, signal_cols, 'Box-Cox -> z')
    check_transform_integrity(intact, intact_bc_t, intact_bc, signal_cols)
    intact_bc.to_csv(OUT_DIR / 'intact_bc.csv', index=False)

### Todos los estímulos por canal

In [ ]:
if intact_bc is not None:
    plot_kde_normalized(intact_bc, features)

### Histogramas

In [ ]:
if intact_bc is not None:
    plot_histograms_by_channel(intact_bc, 'Box-Cox global + Z-score por sujeto')

# λ estimados

Un λ cercano a 0 equivale a un logaritmo; cercano a 1, a no transformar. Se espera que las features de amplitud
(VAR, Energy, Range…) tengan λ bajos y que MNF esté más cerca de 1.

In [ ]:
lam_tablas = {'Yeo-Johnson': tabla_lambdas(pt_yj, signal_cols)}
if pt_bc is not None:
    lam_tablas['Box-Cox'] = tabla_lambdas(pt_bc, signal_cols)

fig, axes = plt.subplots(1, len(lam_tablas), figsize=(7 * len(lam_tablas), 5), squeeze=False)
for ax, (nombre, tabla) in zip(axes[0], lam_tablas.items()):
    sns.heatmap(tabla, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=2, ax=ax)
    ax.set_title(f'λ global – {nombre}')
    ax.set_xlabel('Canal'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

### ¿Hace falta un λ por base de datos?

NinaPro mezcla hardware distinto (Otto Bock en DB1, Delsys en otras, Myo en DB5). Si los λ ajustados por base se alejan mucho del global,
convendría transformar por base en lugar de globalmente. Por sujeto **no** es estimable: solo hay ~8 filas por sujeto.

In [ ]:
def lambdas_por_db(df, cols, method='yeo-johnson', min_filas=30):
    rows = []
    for db, g in df.groupby('origin_db'):
        if len(g) < min_filas:
            print(f'{db}: {len(g)} filas, se omite')
            continue
        pt = PowerTransformer(method=method, standardize=False).fit(g[cols].astype(float).values)
        for col, lam in zip(cols, pt.lambdas_):
            rows.append({'db': db, 'col': col, 'feature': col.split('_', 1)[1], 'lambda': lam})
    return pd.DataFrame(rows)

lam_db = lambdas_por_db(intact, signal_cols, 'yeo-johnson')
lam_glob = pd.Series(pt_yj.lambdas_, index=signal_cols)
lam_db['lambda_global'] = lam_db['col'].map(lam_glob)
lam_db['diff'] = lam_db['lambda'] - lam_db['lambda_global']

tabla_db = lam_db.groupby(['feature', 'db'])['lambda'].median().unstack().loc[features]
tabla_db['GLOBAL'] = lam_db.groupby('feature')['lambda_global'].median().loc[features]

plt.figure(figsize=(10, 5))
sns.heatmap(tabla_db, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=2)
plt.title('Mediana de λ (Yeo-Johnson) por feature: por base vs global')
plt.xlabel(''); plt.ylabel('')
plt.tight_layout(); plt.show()

print('Mediana |λ_db − λ_global| por base:')
print(lam_db.groupby('db')['diff'].apply(lambda s: s.abs().median()).round(3))

In [ ]:
plot_box_before_after(intact, intact_norm, signal_cols, 'Solo Z-score por sujeto')

In [ ]:
plot_box_before_after(intact, intact_yj, signal_cols, 'Yeo-Johnson global + Z-score por sujeto')

# Comparación

In [ ]:
def distribution_metrics(df, cols):
    vals = df[cols]
    return pd.DataFrame({
        'feature': cols,
        'abs_skewness': vals.skew().abs().values,
        'abs_kurtosis': vals.kurt().abs().values,
    })

versiones = {
    'Original': intact,
    'Solo Z-score': intact_norm,
    'Yeo-Johnson (sin z)': intact_yj_t,
    'Yeo-Johnson -> Z-score': intact_yj,
}
if intact_bc is not None:
    versiones['Box-Cox (sin z)'] = intact_bc_t
    versiones['Box-Cox -> Z-score'] = intact_bc

metricas = {k: distribution_metrics(v, signal_cols) for k, v in versiones.items()}

comparison = pd.DataFrame({
    'Transformacion': list(metricas),
    'Mean_Abs_Skewness': [m['abs_skewness'].mean() for m in metricas.values()],
    'Median_Abs_Skewness': [m['abs_skewness'].median() for m in metricas.values()],
    'Mean_Abs_Kurtosis': [m['abs_kurtosis'].mean() for m in metricas.values()],
})
print(comparison.round(4))

- Asimetría (skewness) y curtosis: mejor cuanto más cerca de 0.
- Se incluye la mediana porque la media se ve arrastrada por unas pocas columnas extremas.
- Las filas "sin z" muestran el efecto de la transformación sola; las filas "-> Z-score" son el resultado final del pipeline.

In [ ]:
# Detalle por tipo de feature (resultado final)
detalle = []
for nombre in [k for k in metricas if 'Z-score' in k or k == 'Original']:
    m = metricas[nombre].copy()
    m['tipo'] = m['feature'].str.split('_', n=1).str[1]
    detalle.append(m.groupby('tipo')['abs_skewness'].mean().rename(nombre))
print('|Skewness| medio por tipo de feature:')
print(pd.concat(detalle, axis=1).loc[features].round(3))

# Conclusión

> *Completar después de ejecutar el notebook con los datos reales; los valores de la versión anterior ya no son válidos
> porque provenían de transformaciones ajustadas por sujeto.*

- **Pipeline adoptado:** transformación global (un λ por feature × canal) → z-score por (subject, origin_db).
- **Transformación elegida:** ___ . Criterio: si no hay ceros, Box-Cox y Yeo-Johnson deberían dar resultados muy similares; Yeo-Johnson es preferible
  si alguna columna tiene ceros o si se prevé aplicarlo a datos nuevos que podrían tenerlos.
- **Integridad:** el chequeo de invarianza de Spearman fue ___ (OK / REVISAR).
- **λ por base:** la diferencia mediana entre λ por base y λ global fue ___ ; si alguna base se aleja claramente, considerar transformar por base.
- **Reposo:** excluido del ajuste hasta revisar su extracción de features.

Esta comparación solo evalúa propiedades distribucionales. La decisión final se complementará con el rendimiento en la clasificación de agarres
(notebook 09, LOSO), ajustando el transformador dentro de cada fold solo con los sujetos de entrenamiento.